# 🎬 Director-X Video Server — Colab Edition (V2)

Turns Colab's **free T4 GPU** into a video server for Director-X.

**Auto-selects the best model for your GPU:**
1. Tries **CogVideoX-2B** (highest quality)
2. Falls back to **Wan2.1-1.3B** (great quality, tiny footprint)
3. Last resort: **ModelScope-1.7B** (guaranteed to run)

**Before running:** Runtime → Change runtime type → **T4 GPU**

---

## Step 0: Check GPU

In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"\n✅ {name} — {vram:.0f} GB VRAM")
else:
    print("\n❌ No GPU! Runtime → Change runtime type → T4 GPU")

## Step 1: Install packages

In [ ]:
!pip install -q flask flask-cors pyngrok
!pip install -q diffusers[torch] transformers accelerate sentencepiece protobuf
!pip install -q imageio[ffmpeg] imageio opencv-python-headless safetensors
print("\n✅ Done")

## Step 2: ngrok token
Free at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_AUTH_TOKEN = ""  # @param {type:"string"}

if not NGROK_AUTH_TOKEN:
    print("⚠️  Paste your ngrok auth token above!")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok ready")

## Step 3: Load best available model
Tries CogVideoX-2B first → Wan2.1 → ModelScope. Takes 3-5 min on first run.

In [ ]:
import torch
import gc
import traceback

dtype = torch.float16
device = "cuda"

torch.cuda.empty_cache()
gc.collect()

t2v_pipe = None
active_model = None
export_fn = None

# ─── Try CogVideoX-2B first (higher quality) ───
try:
    print("🎬 Attempting CogVideoX-2B (higher quality)...")
    from diffusers import CogVideoXPipeline
    from diffusers.utils import export_to_video as _export

    t2v_pipe = CogVideoXPipeline.from_pretrained(
        "THUDM/CogVideoX-2b",
        torch_dtype=dtype,
    )
    t2v_pipe.enable_sequential_cpu_offload()
    try:
        t2v_pipe.vae.enable_tiling()
    except Exception:
        pass

    # Quick VRAM test — run a tiny forward pass to see if it fits
    print("   Testing VRAM fit...")
    with torch.inference_mode():
        _test = t2v_pipe(
            prompt="test",
            num_frames=9,
            width=480,
            height=360,
            num_inference_steps=1,
            guidance_scale=6.0,
        ).frames[0]
    del _test
    torch.cuda.empty_cache()
    gc.collect()

    active_model = "CogVideoX-2B"
    export_fn = _export
    print(f"\n✅ CogVideoX-2B loaded! (best quality mode)")
    print(f"   Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB")

except Exception as e:
    print(f"   ⚠️  CogVideoX-2B can't fit: {str(e)[:100]}")
    print("   Switching to Wan2.1 (still good quality, guaranteed to fit)...\n")
    # Clean up failed load
    del t2v_pipe
    t2v_pipe = None
    torch.cuda.empty_cache()
    gc.collect()

# ─── Fallback: Wan2.1-T2V-1.3B (reliable + good quality) ───
if t2v_pipe is None:
    try:
        from diffusers import WanPipeline
        from diffusers.utils import export_to_video as _export

        t2v_pipe = WanPipeline.from_pretrained(
            "Wan-AI/Wan2.1-T2V-1.3B",
            torch_dtype=dtype,
        )
        t2v_pipe.enable_sequential_cpu_offload()

        active_model = "Wan2.1-1.3B"
        export_fn = _export
        print(f"\n✅ Wan2.1-T2V-1.3B loaded!")
        print(f"   VRAM: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

    except Exception as e:
        print(f"\n❌ Wan2.1 also failed: {e}")
        print("   Last resort: trying ModelScope 1.7B...")

        try:
            from diffusers import DiffusionPipeline
            from diffusers.utils import export_to_video as _export

            t2v_pipe = DiffusionPipeline.from_pretrained(
                "ali-vilab/text-to-video-ms-1.7b",
                torch_dtype=dtype,
                variant="fp16",
            )
            t2v_pipe.enable_sequential_cpu_offload()

            active_model = "ModelScope-1.7B"
            export_fn = _export
            print(f"\n✅ ModelScope 1.7B loaded (fallback mode)")

        except Exception as e2:
            print(f"\n❌ All models failed. Error: {e2}")
            raise RuntimeError("No video model could load. Check GPU and dependencies.")

# ─── Model config ───
MODEL_CONFIGS = {
    "CogVideoX-2B": {
        "default_width": 720,
        "default_height": 480,
        "default_frames": 49,
        "default_fps": 8,
        "default_steps": 30,
        "default_guidance": 6.0,
        "max_duration": 6,
        "aspects": {
            "16:9": (720, 480),
            "9:16": (480, 720),
            "1:1":  (480, 480),
            "4:3":  (640, 480),
        },
    },
    "Wan2.1-1.3B": {
        "default_width": 640,
        "default_height": 360,
        "default_frames": 33,
        "default_fps": 16,
        "default_steps": 30,
        "default_guidance": 5.0,
        "max_duration": 5,
        "aspects": {
            "16:9": (640, 360),
            "9:16": (360, 640),
            "1:1":  (480, 480),
            "4:3":  (480, 360),
        },
    },
    "ModelScope-1.7B": {
        "default_width": 512,
        "default_height": 320,
        "default_frames": 24,
        "default_fps": 8,
        "default_steps": 25,
        "default_guidance": 7.5,
        "max_duration": 3,
        "aspects": {
            "16:9": (512, 320),
            "9:16": (320, 512),
            "1:1":  (384, 384),
            "4:3":  (448, 336),
        },
    },
}

cfg = MODEL_CONFIGS[active_model]
print(f"\n📋 Active model: {active_model}")
print(f"   Resolution: {cfg['default_width']}x{cfg['default_height']}")
print(f"   FPS: {cfg['default_fps']} | Max duration: {cfg['max_duration']}s")
print(f"   Aspects: {', '.join(cfg['aspects'].keys())}\n")

## Step 4: Quick test (optional)

In [ ]:
import time

print(f"🎬 Testing {active_model}...")
torch.cuda.empty_cache()
gc.collect()

start = time.time()
with torch.inference_mode():
    test = t2v_pipe(
        prompt="A cinematic slow pan across a dusty frontier town at sunset, golden light",
        num_frames=cfg["default_frames"],
        width=cfg["default_width"],
        height=cfg["default_height"],
        num_inference_steps=cfg["default_steps"],
        guidance_scale=cfg["default_guidance"],
        generator=torch.Generator(device="cuda").manual_seed(42),
    ).frames[0]

export_fn(test, "/content/test.mp4", fps=cfg["default_fps"])
del test
torch.cuda.empty_cache()
gc.collect()

elapsed = time.time() - start
peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"\n✅ {active_model} — {elapsed:.0f}s | Peak VRAM: {peak:.1f} GB")

from IPython.display import HTML
from base64 import b64encode
mp4 = open("/content/test.mp4", "rb").read()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

## Step 5: Start server 🚀
Copy the URL → paste into Director-X → Colab (Local) provider.

In [ ]:
import os
import uuid
import json
import time
import threading
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok
from collections import OrderedDict

app = Flask(__name__)
CORS(app)

OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

jobs = OrderedDict()
MAX_JOBS = 50
gen_queue = []
gen_lock = threading.Lock()
is_generating = False

def get_resolution(aspect_ratio):
    return cfg["aspects"].get(aspect_ratio, (cfg["default_width"], cfg["default_height"]))

def generate_video(job_id, prompt, aspect_ratio="16:9", duration=3, seed=-1):
    global is_generating
    try:
        is_generating = True
        jobs[job_id]["status"] = "generating"
        torch.cuda.empty_cache()
        gc.collect()

        width, height = get_resolution(aspect_ratio)
        fps = cfg["default_fps"]
        num_frames = max(9, min(int(duration * fps) + 1, cfg["default_frames"]))

        gen_kwargs = dict(
            prompt=prompt,
            num_frames=num_frames,
            width=width,
            height=height,
            num_inference_steps=cfg["default_steps"],
            guidance_scale=cfg["default_guidance"],
        )
        # CogVideoX doesn't use negative_prompt the same way
        if active_model != "CogVideoX-2B":
            gen_kwargs["negative_prompt"] = "blurry, low quality, distorted, watermark"
        if seed >= 0:
            gen_kwargs["generator"] = torch.Generator(device="cuda").manual_seed(seed)

        with torch.inference_mode():
            result = t2v_pipe(**gen_kwargs)
            frames = result.frames[0]

        output_path = os.path.join(OUTPUT_DIR, f"{job_id}.mp4")
        export_fn(frames, output_path, fps=fps)

        del frames, result
        torch.cuda.empty_cache()
        gc.collect()

        jobs[job_id]["status"] = "completed"
        jobs[job_id]["video_path"] = output_path
        print(f"\u2705 Job {job_id[:8]} done: {prompt[:50]}...")

    except torch.cuda.OutOfMemoryError:
        jobs[job_id]["status"] = "failed"
        jobs[job_id]["error"] = "GPU out of memory — try shorter duration"
        torch.cuda.empty_cache()
        gc.collect()
        print(f"\u274c Job {job_id[:8]} OOM")
    except Exception as e:
        jobs[job_id]["status"] = "failed"
        jobs[job_id]["error"] = str(e)
        print(f"\u274c Job {job_id[:8]} failed: {e}")
    finally:
        is_generating = False
        torch.cuda.empty_cache()
        gc.collect()
        process_queue()

def process_queue():
    global is_generating
    with gen_lock:
        if is_generating or not gen_queue:
            return
        job = gen_queue.pop(0)
    thread = threading.Thread(target=generate_video, kwargs=job)
    thread.start()

@app.route("/api/health", methods=["GET"])
def health():
    vram_used = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
    vram_total = torch.cuda.get_device_properties(0).total_mem / 1024**3 if torch.cuda.is_available() else 0
    return jsonify({
        "status": "ok",
        "provider": "colab-local",
        "model": active_model,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "vram_used_gb": round(vram_used, 1),
        "vram_total_gb": round(vram_total, 1),
        "queue_length": len(gen_queue),
        "is_generating": is_generating,
        "supported_aspects": list(cfg["aspects"].keys()),
        "max_duration": cfg["max_duration"],
        "resolution": f"{cfg['default_width']}x{cfg['default_height']}",
    })

@app.route("/api/video/submit", methods=["POST"])
def submit_video():
    data = request.json or {}
    prompt = data.get("prompt", "").strip()
    if not prompt:
        return jsonify({"error": "prompt is required"}), 400

    job_id = str(uuid.uuid4())
    aspect_ratio = data.get("aspect_ratio", data.get("aspectRatio", "16:9"))
    duration = min(cfg["max_duration"], max(2, int(data.get("duration", 3))))
    seed = int(data.get("seed", -1))

    jobs[job_id] = {
        "status": "queued",
        "prompt": prompt[:200],
        "aspect_ratio": aspect_ratio,
        "duration": duration,
        "created": time.time(),
        "video_path": None,
        "error": None,
    }

    while len(jobs) > MAX_JOBS:
        old_id, old_job = jobs.popitem(last=False)
        if old_job.get("video_path") and os.path.exists(old_job["video_path"]):
            os.remove(old_job["video_path"])

    gen_queue.append({
        "job_id": job_id,
        "prompt": prompt,
        "aspect_ratio": aspect_ratio,
        "duration": duration,
        "seed": seed,
    })
    process_queue()

    return jsonify({
        "requestId": job_id,
        "statusUrl": f"/api/video/status/{job_id}",
        "provider": "colab-local",
        "model": active_model,
        "queue_position": len(gen_queue),
    })

@app.route("/api/video/status/<job_id>", methods=["GET"])
def video_status(job_id):
    if job_id not in jobs:
        return jsonify({"error": "Job not found"}), 404
    job = jobs[job_id]
    result = {
        "status": job["status"].upper(),
        "prompt": job["prompt"],
        "aspect_ratio": job.get("aspect_ratio"),
    }
    if job["status"] == "completed" and job["video_path"]:
        result["videoUrl"] = f"/api/video/download/{job_id}"
    elif job["status"] == "failed":
        result["error"] = job.get("error", "Unknown error")
    elif job["status"] == "queued":
        pos = next((i for i, j in enumerate(gen_queue) if j["job_id"] == job_id), -1)
        result["queue_position"] = pos + 1 if pos >= 0 else 0
    return jsonify(result)

@app.route("/api/video/download/<job_id>", methods=["GET"])
def download_video(job_id):
    if job_id not in jobs or not jobs[job_id].get("video_path"):
        return jsonify({"error": "Video not found"}), 404
    return send_file(jobs[job_id]["video_path"], mimetype="video/mp4")

@app.route("/api/queue", methods=["GET"])
def queue_info():
    return jsonify({
        "queue_length": len(gen_queue),
        "is_generating": is_generating,
        "active_model": active_model,
        "recent_jobs": [
            {"id": jid, "status": j["status"], "prompt": j["prompt"][:50]}
            for jid, j in list(jobs.items())[-10:]
        ]
    })

port = 5000
public_url = ngrok.connect(port)

print("\n" + "=" * 60)
print("\U0001f3ac DIRECTOR-X VIDEO SERVER IS RUNNING!")
print("=" * 60)
print(f"\n\U0001f916 Model: {active_model}")
print(f"\U0001f4d0 Resolution: {cfg['default_width']}x{cfg['default_height']}")
print(f"\U0001f3a5 FPS: {cfg['default_fps']} | Max clip: {cfg['max_duration']}s")
print(f"\n\U0001f310 Public URL: {public_url}")
print(f"\n\U0001f4cb Paste into Director-X \u2192 Video Provider \u2192 Colab (Local):")
print(f"   {public_url}")
print(f"\n\U0001f527 Health: {public_url}/api/health")
print("\n\u26a1 Aspects: 16:9 (YT), 9:16 (TikTok/Reels), 1:1 (IG), 4:3")
print("\n\u23f3 Keep this notebook running while generating!")
print("=" * 60)

app.run(port=port)

---
## Tips
- **OOM?** Restart runtime, re-run. Model loads from cache (~2 min)
- **Colab free limit:** ~4 hrs GPU/session. Use Kaggle notebook for 30 hrs/week
- **CogVideoX loaded?** You got the best quality. Wan2.1 loaded? Still great — modern 2025 model
